# RL Exit Optimizer

Uses entry signals from `backtest_hawkes_quant.py` as-is.  
Trains a PPO agent to learn **when to exit** each trade.

**Episode** = one trade (entry bar fixed, agent decides hold vs exit each bar)  
**Observation** = 12 market features at the current bar  
**Action** = 0 hold · 1 exit  
**Reward** = sparse (0 while holding, realized P&L % on exit, regret penalty for cutting winners)  

Baseline comparison: original BSI-crossover exit rule.

In [17]:
# !pip install gymnasium stable-baselines3 -q

In [18]:
import sys, os
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import PPO
from stable_baselines3.common.env_checker import check_env
from stable_baselines3.common.monitor import Monitor

from backtest_hawkes_quant import (
    load_ohlc_from_clickhouse,
    compute_hawkes_bsi,
    compute_kama,
    compute_alma,
    generate_signals,
)

In [19]:
def compute_avwap(bars, window=50):
    """
    AVWAP anchored to rolling pivot high and pivot low.
    Pivot high anchor = bar with highest high in last `window` bars (causal).
    Pivot low  anchor = bar with lowest  low  in last `window` bars (causal).
    Resets cumulative TP*vol / vol whenever the anchor bar changes.
    Volume = buyvolume + sellvolume (falls back to uniform if absent).
    """
    highs  = bars['high'].to_numpy(float)
    lows   = bars['low'].to_numpy(float)
    closes = bars['close'].to_numpy(float)

    if 'buyvolume' in bars.columns and 'sellvolume' in bars.columns:
        vols = (bars['buyvolume'].fillna(0) + bars['sellvolume'].fillna(0)).to_numpy(float)
    elif 'volume' in bars.columns:
        vols = bars['volume'].fillna(0).to_numpy(float)
    else:
        vols = np.ones(len(bars), dtype=float)

    n = len(bars)
    avwap_ph = np.full(n, np.nan)
    avwap_pl = np.full(n, np.nan)

    prev_ph = -1;  prev_pl = -1
    ph_tv = 0.0;   ph_v = 0.0
    pl_tv = 0.0;   pl_v = 0.0

    for i in range(n):
        lo  = max(0, i - window + 1)
        tp  = (highs[i] + lows[i] + closes[i]) / 3.0
        vol = vols[i]

        ph = lo + int(np.argmax(highs[lo: i + 1]))
        pl = lo + int(np.argmin(lows[lo:  i + 1]))

        if ph != prev_ph:
            s = slice(ph, i + 1)
            tp_s  = (highs[s] + lows[s] + closes[s]) / 3.0
            ph_tv = float(np.dot(tp_s, vols[s]))
            ph_v  = float(vols[s].sum())
            prev_ph = ph
        else:
            ph_tv += tp * vol;  ph_v += vol

        if pl != prev_pl:
            s = slice(pl, i + 1)
            tp_s  = (highs[s] + lows[s] + closes[s]) / 3.0
            pl_tv = float(np.dot(tp_s, vols[s]))
            pl_v  = float(vols[s].sum())
            prev_pl = pl
        else:
            pl_tv += tp * vol;  pl_v += vol

        if ph_v > 0: avwap_ph[i] = ph_tv / ph_v
        if pl_v > 0: avwap_pl[i] = pl_tv / pl_v

    return avwap_ph, avwap_pl

## 1. Config

In [20]:
# ── Data / indicators ─────────────────────────────────────────────────────
SYMBOL            = 'VN30F1M'
CH_TABLE          = 'ohlc_5m'

KAPPA             = 0.4
QUANTILE_LOOKBACK = 20
Q_LO              = 5.0
Q_HI              = 95.0

MA_TYPE           = 'kama'          # 'kama' | 'alma' | 'none'
KAMA_PERIOD       = 10
KAMA_FAST         = 2
KAMA_SLOW         = 30
ALMA_WINDOW       = 9
ALMA_OFFSET       = 0.85
ALMA_SIGMA        = 6.0

SL_BARS           = 10
CALM_BARS         = 5
CALM_THRESHOLD    = 500.0

AVWAP_SHORT       = 50    # pivot window for short-term AVWAP (bars)
AVWAP_LONG        = 200   # pivot window for long-term  AVWAP (bars)

# ── RL environment ────────────────────────────────────────────────────────
MAX_HOLD          = 60
HOLDING_COST      = 0.0   # no per-bar penalty
REGRET_FACTOR     = 0.5   # penalty weight for agent exits below the trade's peak upnl

# ── Train / test split ────────────────────────────────────────────────────
TRAIN_END         = '2026-01-31'

# ── PPO training ─────────────────────────────────────────────────────────
TOTAL_TIMESTEPS   = 500_000
SEED              = 42

## 2. Load data & compute indicators

In [21]:
bars = load_ohlc_from_clickhouse(SYMBOL, table=CH_TABLE)

bars = compute_hawkes_bsi(
    bars, kappa=KAPPA, quantile_lookback=QUANTILE_LOOKBACK,
    q_lo_pct=Q_LO, q_hi_pct=Q_HI,
)

if MA_TYPE == 'kama':
    bars = compute_kama(bars, period=KAMA_PERIOD, fast=KAMA_FAST, slow=KAMA_SLOW)
elif MA_TYPE == 'alma':
    bars = compute_alma(bars, window=ALMA_WINDOW, offset=ALMA_OFFSET, sigma=ALMA_SIGMA)
    bars['kama'] = bars['alma']

bars = bars.copy()
avwap_ph_s, avwap_pl_s = compute_avwap(bars, window=AVWAP_SHORT)
avwap_ph_l, avwap_pl_l = compute_avwap(bars, window=AVWAP_LONG)
bars['avwap_ph_s'] = avwap_ph_s
bars['avwap_pl_s'] = avwap_pl_s
bars['avwap_ph_l'] = avwap_ph_l
bars['avwap_pl_l'] = avwap_pl_l

bars = bars.reset_index(drop=True)
print(f'Bars loaded: {len(bars):,}  ({bars["stamp"].iloc[0]}  →  {bars["stamp"].iloc[-1]})')
bars[['stamp','open','high','low','close','bsi','q_lo','q_hi',
      'avwap_ph_s','avwap_pl_s','avwap_ph_l','avwap_pl_l']].tail(3)

2026-04-27 18:24:07 [INFO] Loaded 12088 OHLC rows from default.ohlc_5m (VN30F1M)


Bars loaded: 12,088  (2025-05-05 09:00:00  →  2026-04-24 14:45:00)


,stamp,open,high,low,close,bsi,q_lo,q_hi,avwap_ph_s,avwap_pl_s,avwap_ph_l,avwap_pl_l
12085,2026-04-24 14:20:00,2012.0,2015.0,2010.3,2012.9,543.045011,-1816.446528,2541.027417,2011.269821,2010.508318,2018.209209,2016.558308
12086,2026-04-24 14:25:00,2012.3,2015.5,2010.0,2013.0,943.013957,-1816.446528,2541.027417,2011.306831,2010.664268,2018.152293,2016.528685
12087,2026-04-24 14:45:00,2015.0,2015.0,2015.0,2015.0,4861.121159,-1816.446528,2813.044634,2010.837827,2010.865535,2018.128255,2016.519913


## 3. Generate entries & build trade list

In [22]:
long_entries, short_entries, _, _ = generate_signals(
    bars,
    allow_short    = True,
    use_kama_gate  = MA_TYPE != 'none',
    sl_bars        = SL_BARS,
    calm_bars      = CALM_BARS,
    calm_threshold = CALM_THRESHOLD,
)

opens  = bars['open'].to_numpy(float)
highs  = bars['high'].to_numpy(float)
lows   = bars['low'].to_numpy(float)
n_bars = len(bars)

trades = []
for i in range(n_bars):
    if long_entries[i] or short_entries[i]:
        direction = +1 if long_entries[i] else -1
        win = max(0, (i - 1) - SL_BARS + 1)
        sl_price = (
            float(np.min(lows[win: i]))   if direction == +1 else
            float(np.max(highs[win: i]))
        )
        trades.append({
            'entry_bar'  : i,
            'direction'  : direction,
            'entry_price': float(opens[i]),
            'sl_price'   : sl_price,
            'stamp'      : bars['stamp'].iloc[i],
        })

print(f'Total entries: {len(trades)}')
print(f'  Long : {sum(t["direction"]==+1 for t in trades)}')
print(f'  Short: {sum(t["direction"]==-1 for t in trades)}')

Total entries: 356
  Long : 180
  Short: 176


## 4. Train / test split

In [23]:
cutoff = pd.Timestamp(TRAIN_END)
train_trades = [t for t in trades if pd.Timestamp(t['stamp']) <= cutoff]
test_trades  = [t for t in trades if pd.Timestamp(t['stamp']) >  cutoff]

print(f'Train trades: {len(train_trades)}  (up to {TRAIN_END})')
print(f'Test  trades: {len(test_trades)}')

if len(train_trades) < 20:
    print('\n⚠  Very few training trades. Adjust TRAIN_END or strategy params.')

Train trades: 280  (up to 2026-01-31)
Test  trades: 76


## 5. Baseline: rule-based exit

Original strategy: long exits when `BSI < q_lo`, short exits when `BSI >= q_hi`.  
SL active for first `SL_BARS` bars.  
Forced exit at `MAX_HOLD` bars.

In [24]:
def simulate_rule_exits(bars_df, trade_list, max_hold=MAX_HOLD, sl_bars=SL_BARS):
    """Reproduce the original BSI-crossover exit rule for a list of trades."""
    bsi    = bars_df['bsi'].to_numpy(float)
    q_lo   = bars_df['q_lo'].to_numpy(float)
    q_hi   = bars_df['q_hi'].to_numpy(float)
    closes = bars_df['close'].to_numpy(float)
    highs  = bars_df['high'].to_numpy(float)
    lows   = bars_df['low'].to_numpy(float)
    n      = len(bars_df)

    records = []
    for t in trade_list:
        eb    = t['entry_bar']
        ep    = t['entry_price']
        d     = t['direction']
        sl    = t.get('sl_price', np.nan)

        exit_bar  = min(eb + max_hold, n - 1)
        exit_type = 'max_hold'

        for i in range(eb + 1, min(eb + max_hold + 1, n)):
            bars_held = i - eb
            if not np.isnan(sl) and bars_held <= sl_bars:
                if d == +1 and lows[i]  <= sl: exit_bar = i; exit_type = 'sl'; break
                if d == -1 and highs[i] >= sl: exit_bar = i; exit_type = 'sl'; break
            if d == +1 and bsi[i] <  q_lo[i]: exit_bar = i; exit_type = 'bsi'; break
            if d == -1 and bsi[i] >= q_hi[i]: exit_bar = i; exit_type = 'bsi'; break

        pnl = (closes[exit_bar] - ep) / ep * d
        records.append({
            'entry_bar' : eb,
            'exit_bar'  : exit_bar,
            'direction' : d,
            'pnl'       : pnl,
            'bars_held' : exit_bar - eb,
            'exit_type' : exit_type,
        })
    return pd.DataFrame(records)


def print_stats(name, df):
    if df.empty: print(f'{name}: no trades'); return
    wr = (df['pnl'] > 0).mean()
    print(f'{name:25s}  n={len(df):4d}  '
          f'total={df["pnl"].sum()*100:+7.3f}%  '
          f'mean={df["pnl"].mean()*100:+6.4f}%  '
          f'wr={wr:.1%}  '
          f'avg_hold={df["bars_held"].mean():.1f}')


baseline_train = simulate_rule_exits(bars, train_trades)
baseline_test  = simulate_rule_exits(bars, test_trades)

print_stats('Baseline  [TRAIN]', baseline_train)
print_stats('Baseline  [TEST] ', baseline_test)

Baseline  [TRAIN]          n= 280  total=+38.911%  mean=+0.1390%  wr=47.9%  avg_hold=13.9
Baseline  [TEST]           n=  76  total=+11.756%  mean=+0.1547%  wr=40.8%  avg_hold=15.6


## 6. RL Environment

### Observation (12 features) — all P&L / return / distance features in **%**
| # | Feature | Description |
|---|---------|-------------|
| 0 | `upnl` | Unrealized P&L % (direction-adjusted) |
| 1 | `bars_held_norm` | bars held / MAX_HOLD |
| 2 | `bsi_band_pos` | (BSI − q_lo) / (q_hi − q_lo), clipped \[-2, 3\] |
| 3 | `gate_dist` | (close − gate_MA) / close × direction × 100 % |
| 4 | `ret_1` | 1-bar return × direction × 100 % |
| 5 | `ret_5` | 5-bar return × direction × 100 % |
| 6 | `sl_active` | 1 while within SL window, else 0 |
| 7 | `drawdown` | (peak_upnl − upnl) in % since entry |
| 8 | `avwap_ph_s_dist` | (close − AVWAP_pivot-high_50) / close × direction × 100 % |
| 9 | `avwap_pl_s_dist` | (close − AVWAP_pivot-low_50)  / close × direction × 100 % |
| 10 | `avwap_ph_l_dist` | (close − AVWAP_pivot-high_200) / close × direction × 100 % |
| 11 | `avwap_pl_l_dist` | (close − AVWAP_pivot-low_200)  / close × direction × 100 % |

### Reward (all in %)
- **Hold**: `0` (sparse — avoids per-bar noise biasing toward early exit)
- **SL / forced exit**: `realized_pnl × 100` (terminal)
- **Agent exit**: `realized_pnl × 100 − REGRET_FACTOR × max(0, peak_upnl − upnl_at_exit)`
  - Penalizes cutting a winner below its in-trade peak; losers and SL/forced exits are exempt

In [25]:
N_FEATURES = 12

def _avwap_dist(close, avwap_val, direction):
    if np.isnan(avwap_val):
        return 0.0
    return (close - avwap_val) / close * direction * 100.0


class TradeExitEnv(gym.Env):
    metadata = {'render_modes': []}

    def __init__(self, bars_df, trade_list, max_hold=MAX_HOLD,
                 sl_bars=SL_BARS, holding_cost=HOLDING_COST,
                 regret_factor=REGRET_FACTOR):
        super().__init__()
        self._bars         = bars_df.reset_index(drop=True)
        self._trades       = trade_list
        self.max_hold      = max_hold
        self.sl_bars       = sl_bars
        self.holding_cost  = holding_cost
        self.regret_factor = regret_factor
        self._n            = len(bars_df)

        self._closes     = self._bars['close'].to_numpy(float)
        self._highs      = self._bars['high'].to_numpy(float)
        self._lows       = self._bars['low'].to_numpy(float)
        self._bsi        = self._bars['bsi'].to_numpy(float)
        self._q_lo       = self._bars['q_lo'].to_numpy(float)
        self._q_hi       = self._bars['q_hi'].to_numpy(float)
        self._kama       = (self._bars['kama'].to_numpy(float)
                            if 'kama' in self._bars.columns else None)
        self._avwap_ph_s = self._bars['avwap_ph_s'].to_numpy(float)
        self._avwap_pl_s = self._bars['avwap_pl_s'].to_numpy(float)
        self._avwap_ph_l = self._bars['avwap_ph_l'].to_numpy(float)
        self._avwap_pl_l = self._bars['avwap_pl_l'].to_numpy(float)

        self.observation_space = spaces.Box(
            low=-np.inf, high=np.inf, shape=(N_FEATURES,), dtype=np.float32,
        )
        self.action_space = spaces.Discrete(2)  # 0=hold, 1=exit

        self._entry_bar   = 0
        self._cur         = 0
        self._entry_price = 1.0
        self._direction   = 1
        self._sl_price    = np.nan
        self._peak_upnl   = 0.0

    def _upnl_pct(self, bar_idx):
        c = self._closes[min(bar_idx, self._n - 1)]
        return (c - self._entry_price) / self._entry_price * self._direction * 100.0

    def _obs(self):
        i = min(self._cur, self._n - 1)
        c = self._closes[i]

        upnl_pct       = self._upnl_pct(i)
        bars_held      = i - self._entry_bar
        bars_held_norm = bars_held / self.max_hold

        q_lo, q_hi = self._q_lo[i], self._q_hi[i]
        band = q_hi - q_lo
        bsi_band_pos = float(np.clip(
            (self._bsi[i] - q_lo) / band if band > 1e-10 else 0.0, -2.0, 3.0))

        gate_dist = (
            (c - self._kama[i]) / c * self._direction * 100.0
            if self._kama is not None and not np.isnan(self._kama[i]) else 0.0
        )
        ret1 = (c / self._closes[max(0, i-1)] - 1.0) * self._direction * 100.0
        ret5 = (c / self._closes[max(0, i-5)] - 1.0) * self._direction * 100.0
        sl_active = 1.0 if bars_held < self.sl_bars else 0.0
        drawdown  = max(0.0, self._peak_upnl - upnl_pct)

        d = self._direction
        return np.array([
            upnl_pct, bars_held_norm, bsi_band_pos, gate_dist,
            ret1, ret5, sl_active, drawdown,
            _avwap_dist(c, self._avwap_ph_s[i], d),
            _avwap_dist(c, self._avwap_pl_s[i], d),
            _avwap_dist(c, self._avwap_ph_l[i], d),
            _avwap_dist(c, self._avwap_pl_l[i], d),
        ], dtype=np.float32)

    def _check_sl(self, i):
        bars_held = i - self._entry_bar
        if np.isnan(self._sl_price) or bars_held > self.sl_bars:
            return False
        if self._direction == +1 and self._lows[i]  <= self._sl_price: return True
        if self._direction == -1 and self._highs[i] >= self._sl_price: return True
        return False

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        t = self._trades[self.np_random.integers(0, len(self._trades))]
        self._entry_bar   = t['entry_bar']
        self._cur         = t['entry_bar']
        self._entry_price = t['entry_price']
        self._direction   = t['direction']
        self._sl_price    = t.get('sl_price', np.nan)
        self._peak_upnl   = 0.0
        return self._obs(), {}

    def step(self, action):
        i = min(self._cur, self._n - 1)
        bars_held = i - self._entry_bar
        sl_hit    = self._check_sl(i)
        forced    = bars_held >= self.max_hold or i >= self._n - 1

        if action == 1 or sl_hit or forced:
            exit_price = (
                self._sl_price if sl_hit and not np.isnan(self._sl_price)
                else self._closes[i]
            )
            pnl = (exit_price - self._entry_price) / self._entry_price * self._direction
            reward = float(pnl * 100.0)

            # Penalize voluntary exits that cut a winning trade below its peak.
            # SL and forced exits are exempt — those aren't premature choices.
            if action == 1 and not sl_hit and not forced and self.regret_factor > 0:
                upnl_at_exit = float(pnl * 100.0)
                regret = max(0.0, self._peak_upnl - upnl_at_exit)
                reward -= self.regret_factor * regret

            return self._obs(), reward, True, False, {
                'pnl': pnl, 'bars_held': bars_held,
                'exit': 'sl' if sl_hit else ('forced' if forced else 'agent'),
            }

        self._cur += 1
        self._peak_upnl = max(self._peak_upnl, self._upnl_pct(self._cur))
        return self._obs(), 0.0, False, False, {'pnl': 0.0, 'bars_held': bars_held}

    def render(self): pass


if train_trades:
    _env = TradeExitEnv(bars, train_trades)
    check_env(_env, warn=True)
    print('Environment check passed.')

Environment check passed.


## 7. Train PPO agent

In [ ]:
train_env = Monitor(TradeExitEnv(bars, train_trades))

eval_trades = test_trades if len(test_trades) >= 10 else train_trades
eval_env    = Monitor(TradeExitEnv(bars, eval_trades))

model = PPO(
    'MlpPolicy',
    train_env,
    verbose          = 0,
    n_steps          = 1024,
    batch_size       = 64,
    n_epochs         = 10,
    gamma            = 0.99,
    learning_rate    = 3e-4,
    ent_coef         = 0.05,
    seed             = SEED,
    policy_kwargs    = dict(net_arch=[64, 64]),
)

model.learn(total_timesteps=TOTAL_TIMESTEPS, progress_bar=True)
print('Training complete.')

Output()

Using cpu device
Wrapping the env in a DummyVecEnv.


---------------------------------
| rollout/           |          |
|    ep_len_mean     | 2.1      |
|    ep_rew_mean     | -0.0403  |
| time/              |          |
|    fps             | 7450     |
|    iterations      | 1        |
|    time_elapsed    | 0        |
|    total_timesteps | 1024     |
---------------------------------


------------------------------------------
| rollout/                |              |
|    ep_len_mean          | 1.94         |
|    ep_rew_mean          | -0.0492      |
| time/                   |              |
|    fps                  | 4881         |
|    iterations           | 2            |
|    time_elapsed         | 0            |
|    total_timesteps      | 2048         |
| train/                  |              |
|    approx_kl            | 0.0096169375 |
|    clip_fraction        | 0.03         |
|    clip_range           | 0.2          |
|    entropy_loss         | -0.688       |
|    explained_variance   | -1.18        |
|    learning_rate        | 0.0003       |
|    loss                 | -0.00715     |
|    n_updates            | 10           |
|    policy_gradient_loss | -0.00513     |
|    value_loss           | 0.0665       |
------------------------------------------


-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 1.64        |
|    ep_rew_mean          | -0.0428     |
| time/                   |             |
|    fps                  | 4371        |
|    iterations           | 3           |
|    time_elapsed         | 0           |
|    total_timesteps      | 3072        |
| train/                  |             |
|    approx_kl            | 0.004649585 |
|    clip_fraction        | 0.00801     |
|    clip_range           | 0.2         |
|    entropy_loss         | -0.685      |
|    explained_variance   | 0.583       |
|    learning_rate        | 0.0003      |
|    loss                 | -0.0169     |
|    n_updates            | 20          |
|    policy_gradient_loss | -0.00682    |
|    value_loss           | 0.0307      |
-----------------------------------------


...

----------------------------------------
| rollout/                |            |
|    ep_len_mean          | 11.5       |
|    ep_rew_mean          | 0.153      |
| time/                   |            |
|    fps                  | 3559       |
|    iterations           | 149        |
|    time_elapsed         | 42         |
|    total_timesteps      | 152576     |
| train/                  |            |
|    approx_kl            | 0.00487473 |
|    clip_fraction        | 0.0402     |
|    clip_range           | 0.2        |
|    entropy_loss         | -0.106     |
|    explained_variance   | 0.749      |
|    learning_rate        | 0.0003     |
|    loss                 | 0.0147     |
|    n_updates            | 1480       |
|    policy_gradient_loss | -0.00482   |
|    value_loss           | 0.0627     |
----------------------------------------


------------------------------------------
| rollout/                |              |
|    ep_len_mean          | 13.1         |
|    ep_rew_mean          | 0.118        |
| time/                   |              |
|    fps                  | 3559         |
|    iterations           | 150          |
|    time_elapsed         | 43           |
|    total_timesteps      | 153600       |
| train/                  |              |
|    approx_kl            | 0.0035021962 |
|    clip_fraction        | 0.0298       |
|    clip_range           | 0.2          |
|    entropy_loss         | -0.114       |
|    explained_variance   | 0.748        |
|    learning_rate        | 0.0003       |
|    loss                 | 0.0173       |
|    n_updates            | 1490         |
|    policy_gradient_loss | -0.00674     |
|    value_loss           | 0.0657       |
------------------------------------------


------------------------------------------
| rollout/                |              |
|    ep_len_mean          | 12.3         |
|    ep_rew_mean          | 0.132        |
| time/                   |              |
|    fps                  | 3558         |
|    iterations           | 151          |
|    time_elapsed         | 43           |
|    total_timesteps      | 154624       |
| train/                  |              |
|    approx_kl            | 0.0012873886 |
|    clip_fraction        | 0.0129       |
|    clip_range           | 0.2          |
|    entropy_loss         | -0.0872      |
|    explained_variance   | 0.779        |
|    learning_rate        | 0.0003       |
|    loss                 | 0.0226       |
|    n_updates            | 1500         |
|    policy_gradient_loss | -0.00199     |
|    value_loss           | 0.0616       |
------------------------------------------


------------------------------------------
| rollout/                |              |
|    ep_len_mean          | 10.7         |
|    ep_rew_mean          | 0.15         |
| time/                   |              |
|    fps                  | 3559         |
|    iterations           | 152          |
|    time_elapsed         | 43           |
|    total_timesteps      | 155648       |
| train/                  |              |
|    approx_kl            | 0.0023158423 |
|    clip_fraction        | 0.0231       |
|    clip_range           | 0.2          |
|    entropy_loss         | -0.0925      |
|    explained_variance   | 0.8          |
|    learning_rate        | 0.0003       |
|    loss                 | 0.0275       |
|    n_updates            | 1510         |
|    policy_gradient_loss | -0.00258     |
|    value_loss           | 0.0683       |
------------------------------------------


-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 11.2        |
|    ep_rew_mean          | 0.0799      |
| time/                   |             |
|    fps                  | 3561        |
|    iterations           | 153         |
|    time_elapsed         | 43          |
|    total_timesteps      | 156672      |
| train/                  |             |
|    approx_kl            | 0.008234719 |
|    clip_fraction        | 0.0542      |
|    clip_range           | 0.2         |
|    entropy_loss         | -0.13       |
|    explained_variance   | 0.737       |
|    learning_rate        | 0.0003      |
|    loss                 | 0.0144      |
|    n_updates            | 1520        |
|    policy_gradient_loss | -0.00548    |
|    value_loss           | 0.0576      |
-----------------------------------------


In [ ]:
MODEL_PATH = f'rl_exit_{SYMBOL}.zip'
model.save(MODEL_PATH)
print(f'Model saved → {MODEL_PATH}')

# To reload later:
# model = PPO.load(MODEL_PATH)

## 8. Evaluate RL agent

In [ ]:
def evaluate_agent(model, bars_df, trade_list, max_hold=MAX_HOLD, sl_bars=SL_BARS):
    """Run the trained agent deterministically on every trade in trade_list."""
    env     = TradeExitEnv(bars_df, trade_list, max_hold=max_hold, sl_bars=sl_bars)
    records = []

    for t in trade_list:
        env._entry_bar   = t['entry_bar']
        env._cur         = t['entry_bar']
        env._entry_price = t['entry_price']
        env._direction   = t['direction']
        env._sl_price    = t.get('sl_price', np.nan)
        env._peak_upnl   = 0.0

        obs = env._obs()
        done = False
        info = {}
        while not done:
            action, _ = model.predict(obs, deterministic=True)
            obs, _, terminated, truncated, info = env.step(int(action))
            done = terminated or truncated

        records.append({
            'entry_bar' : t['entry_bar'],
            'exit_bar'  : env._cur,
            'direction' : t['direction'],
            'pnl'       : info.get('pnl', 0.0),
            'bars_held' : info.get('bars_held', 0),
            'exit_type' : info.get('exit', '?'),
        })

    return pd.DataFrame(records)


rl_train = evaluate_agent(model, bars, train_trades)
rl_test  = evaluate_agent(model, bars, test_trades)

print('─' * 70)
print_stats('Baseline  [TRAIN]', baseline_train)
print_stats('RL agent  [TRAIN]', rl_train)
print('─' * 70)
print_stats('Baseline  [TEST] ', baseline_test)
print_stats('RL agent  [TEST] ', rl_test)
print('─' * 70)

In [ ]:
def full_stats(df):
    if df.empty:
        return {}
    pnl    = df['pnl']
    wins   = pnl[pnl > 0]
    losses = pnl[pnl < 0]

    cum      = pnl.cumsum()
    max_dd   = (cum.cummax() - cum).max()
    pf       = (wins.sum() / abs(losses.sum())) if losses.sum() != 0 else np.inf
    mean_    = pnl.mean()
    std_     = pnl.std(ddof=1)
    neg_std  = losses.std(ddof=1) if len(losses) > 1 else np.nan
    sharpe   = mean_ / std_    if std_  > 0 else np.nan
    sortino  = mean_ / neg_std if (neg_std and neg_std > 0) else np.nan

    return {
        'n trades'        : len(df),
        'total P&L %'     : pnl.sum()   * 100,
        'mean P&L %'      : mean_        * 100,
        'median P&L %'    : pnl.median() * 100,
        'win rate'        : (pnl > 0).mean(),
        'profit factor'   : pf,
        'avg winner %'    : wins.mean()   * 100 if len(wins)   else np.nan,
        'avg loser %'     : losses.mean() * 100 if len(losses) else np.nan,
        'biggest winner %': pnl.max()     * 100,
        'biggest loser %' : pnl.min()     * 100,
        'max drawdown %'  : max_dd        * 100,
        'sharpe (trade)'  : sharpe,
        'sortino (trade)' : sortino,
        'avg hold bars'   : df['bars_held'].mean(),
    }


def compare_stats(base_df, rl_df, label='TEST'):
    b = full_stats(base_df)
    r = full_stats(rl_df)
    if not b or not r:
        print('No data.')
        return

    fmt = {
        'n trades'        : '{:.0f}',
        'total P&L %'     : '{:+.3f}%',
        'mean P&L %'      : '{:+.4f}%',
        'median P&L %'    : '{:+.4f}%',
        'win rate'        : '{:.1%}',
        'profit factor'   : '{:.3f}',
        'avg winner %'    : '{:+.4f}%',
        'avg loser %'     : '{:+.4f}%',
        'biggest winner %': '{:+.4f}%',
        'biggest loser %' : '{:+.4f}%',
        'max drawdown %'  : '{:.3f}%',
        'sharpe (trade)'  : '{:.3f}',
        'sortino (trade)' : '{:.3f}',
        'avg hold bars'   : '{:.1f}',
    }

    rows = []
    for k, f in fmt.items():
        bv, rv = b.get(k, np.nan), r.get(k, np.nan)
        try:   b_str = f.format(bv)
        except: b_str = str(bv)
        try:   r_str = f.format(rv)
        except: r_str = str(rv)
        rows.append({'metric': k, 'Baseline': b_str, 'RL agent': r_str})

    tbl = pd.DataFrame(rows).set_index('metric')
    print(f'\n{"─"*50}  {label}  {"─"*50}')
    print(tbl.to_string())
    print('─' * 105)
    return tbl


tbl_test = compare_stats(baseline_test, rl_test, label='TEST')
_        = compare_stats(baseline_train, rl_train, label='TRAIN')

## 9. Visualization

In [ ]:
def plot_comparison(base_df, rl_df, title=''):
    if base_df.empty or rl_df.empty:
        print('No trades to plot.')
        return

    fig = plt.figure(figsize=(14, 10))
    gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.4, wspace=0.35)

    ax1 = fig.add_subplot(gs[0, :])
    ax1.plot(base_df['pnl'].cumsum() * 100, label='Baseline (BSI exit)', color='steelblue')
    ax1.plot(rl_df['pnl'].cumsum()   * 100, label='RL agent',            color='darkorange')
    ax1.axhline(0, color='gray', linewidth=0.8, linestyle='--')
    ax1.set_xlabel('Trade #'); ax1.set_ylabel('Cumulative P&L (%)')
    ax1.set_title('Cumulative P&L — Baseline vs RL'); ax1.legend(); ax1.grid(alpha=0.3)

    ax2 = fig.add_subplot(gs[1, 0])
    bins = np.linspace(
        min(base_df['pnl'].min(), rl_df['pnl'].min()),
        max(base_df['pnl'].max(), rl_df['pnl'].max()), 30,
    ) * 100
    ax2.hist(base_df['pnl'] * 100, bins=bins, alpha=0.5, label='Baseline', color='steelblue')
    ax2.hist(rl_df['pnl']   * 100, bins=bins, alpha=0.5, label='RL',       color='darkorange')
    ax2.axvline(0, color='gray', linewidth=0.8)
    ax2.set_xlabel('P&L (%)'); ax2.set_ylabel('Count')
    ax2.set_title('P&L Distribution'); ax2.legend(); ax2.grid(alpha=0.3)

    ax3 = fig.add_subplot(gs[1, 1])
    max_bh  = max(base_df['bars_held'].max(), rl_df['bars_held'].max())
    bh_bins = np.arange(0, max_bh + 2)
    ax3.hist(base_df['bars_held'], bins=bh_bins, alpha=0.5, label='Baseline', color='steelblue')
    ax3.hist(rl_df['bars_held'],   bins=bh_bins, alpha=0.5, label='RL',       color='darkorange')
    ax3.set_xlabel('Bars held'); ax3.set_ylabel('Count')
    ax3.set_title('Hold Duration'); ax3.legend(); ax3.grid(alpha=0.3)

    fig.suptitle(title or 'Baseline vs RL Exit — Test Set', fontsize=13, y=1.01)
    plt.tight_layout(); plt.show()


plot_comparison(baseline_test, rl_test, title=f'{SYMBOL} — Test set')

In [ ]:
if not rl_test.empty:
    print('RL exit types [TEST]:')
    print(rl_test['exit_type'].value_counts().to_string())
    print()
    print('RL mean P&L by exit type:')
    print((rl_test.groupby('exit_type')['pnl'].mean() * 100).map('{:+.4f}%'.format).to_string())

## 10. Inspect individual trades

In [ ]:
def plot_trade(bars_df, trade, rl_exit_bar, base_exit_bar, window_extra=10):
    eb    = trade['entry_bar']
    start = max(0, eb - 5)
    end   = min(len(bars_df), max(rl_exit_bar, base_exit_bar) + window_extra)
    sl    = bars_df.iloc[start:end]

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True,
                                    gridspec_kw={'height_ratios': [2, 1]})
    x  = np.arange(len(sl))
    ei = eb - start
    ri = rl_exit_bar   - start
    bi = base_exit_bar - start

    ax1.plot(x, sl['close'].values, color='white', linewidth=0.8, label='close', alpha=0.7)
    if 'kama' in bars_df.columns:
        ax1.plot(x, sl['kama'].values, color='gold', linewidth=1, linestyle='--', label='Gate MA')

    col = 'lime' if trade['direction'] == +1 else 'red'
    ax1.axvline(ei, color=col,      linestyle='-',  linewidth=1.5, label='Entry')
    ax1.axvline(ri, color='orange', linestyle='--', linewidth=1.5, label='RL exit')
    ax1.axvline(bi, color='cyan',   linestyle=':',  linewidth=1.5, label='Baseline exit')
    ax1.set_ylabel('Price'); ax1.legend(fontsize=8); ax1.set_facecolor('#1a1a2e')

    ax2.plot(x, sl['bsi'].values,  color='#10a4f4',       linewidth=0.8, label='BSI')
    ax2.plot(x, sl['q_lo'].values, color='mediumseagreen', linewidth=0.8, linestyle='--')
    ax2.plot(x, sl['q_hi'].values, color='salmon',         linewidth=0.8, linestyle='--')
    ax2.axvline(ei, color=col,      linestyle='-',  linewidth=1.5)
    ax2.axvline(ri, color='orange', linestyle='--', linewidth=1.5)
    ax2.axvline(bi, color='cyan',   linestyle=':',  linewidth=1.5)
    ax2.set_ylabel('Hawkes BSI'); ax2.set_facecolor('#1a1a2e')

    d_str = 'LONG' if trade['direction'] == +1 else 'SHORT'
    fig.suptitle(f'Trade {d_str} @ bar {eb}', fontsize=11)
    plt.tight_layout(); plt.show()


if not baseline_test.empty and not rl_test.empty:
    merged_test = baseline_test[['entry_bar','pnl','exit_bar']].rename(
        columns={'pnl':'pnl_base','exit_bar':'exit_base'}).merge(
        rl_test[['entry_bar','pnl','exit_bar']].rename(
            columns={'pnl':'pnl_rl','exit_bar':'exit_rl'}),
        on='entry_bar',
    )
    merged_test['diff'] = merged_test['pnl_rl'] - merged_test['pnl_base']
    top = merged_test.nlargest(3, 'diff')

    for _, row in top.iterrows():
        t = next(t for t in test_trades if t['entry_bar'] == row['entry_bar'])
        plot_trade(bars, t, int(row['exit_rl']), int(row['exit_base']))